# Week 7 - Evaluation

In [1]:
import os
os.environ["PYTHONUTF8"] = "1"

import re
import sys
sys.path.insert(0, "../src")

import time
import json
import warnings
warnings.filterwarnings('ignore')

import pandas as pd
from dotenv import load_dotenv
load_dotenv('../.env')

os.makedirs('../data/eval', exist_ok=True)
os.makedirs('../results', exist_ok=True)

In [2]:
# pymupdf4llm 활용해 만든 VectorDB 호출

from langchain_chroma import Chroma
from langchain_openai import OpenAIEmbeddings

BASE_DIR = "C:/Users/seohyun/OneDrive/2026/Advanced_RAG"
PDF_PATH = os.path.join(BASE_DIR, 'data', 'registration_of_real_estatee_manual.pdf')
DENSE_DB_PATH = os.path.join(BASE_DIR, 'chroma_db', 'real_estatee_manual')
COLLECTION_NAME = 'real_estatee_manual'

embeddings = OpenAIEmbeddings(model='text-embedding-3-large')

db = Chroma(
    persist_directory=DENSE_DB_PATH,
    embedding_function=embeddings,
    collection_name=COLLECTION_NAME,
)
print(f'기존 ChromaDB 로드: {db._collection.count()}개 문서')

기존 ChromaDB 로드: 327개 문서


In [3]:
# 변경된 임베딩으로 Hybrid Search+Reranking 실험

from typing import List, Any
from sentence_transformers import CrossEncoder
from langchain_core.documents import Document
from langchain_community.retrievers import BM25Retriever
from langchain_classic.retrievers import EnsembleRetriever


# BM25용 Document 리스트 생성
raw = db.get(include=["documents", "metadatas"])

bm25_docs = [
    Document(page_content=doc, metadata=metadata or {})
    for doc, metadata in zip(raw["documents"], raw["metadatas"])
]


# Cross-Encoder Re-Ranker
def korean_tokenizer(text: str):
    """BM25용 한국어 토크나이저: 특수문자 제거 + 공백 분리 + 1글자 제거"""
    cleaned = re.sub("[^가-힣a-zA-Z0-9]", " ", text)
    return [t for t in cleaned.split() if len(t) > 1]

RERANKER_MODEL = "BAAI/bge-reranker-v2-m3"
cross_encoder = CrossEncoder(RERANKER_MODEL)


# Retriever 
bm25_retriever_k20 = BM25Retriever.from_documents(
    bm25_docs,
    k=10,
    preprocess_func=korean_tokenizer,
)

dense_retriever_k20 = db.as_retriever(
    search_type="mmr",
    search_kwargs={"k": 10, "fetch_k": 20},
)

hybrid_retriever_k20 = EnsembleRetriever(
    retrievers=[bm25_retriever_k20, dense_retriever_k20],
    weights=[0.5, 0.5],
    c=60,
)

def hybrid_rerank_retriever(query: str, top_k: int = 5) -> List[Document]:
    """Hybrid 1차 검색 -> Cross-Encoder 재정렬"""
    candidates = hybrid_retriever_k20.invoke(query)

    if not candidates:
        return []

    pairs = [(query, doc.page_content) for doc in candidates]
    scores = cross_encoder.predict(pairs)

    ranked = sorted(zip(scores, candidates), key=lambda x: x[0], reverse=True)
    return [doc for _, doc in ranked[:top_k]]

Loading weights: 100%|██████████| 393/393 [00:00<00:00, 1259.98it/s]


In [4]:
# 비교 실험

import time
import json
from langchain_openai import ChatOpenAI
from langchain_core.prompts import ChatPromptTemplate

from prompt.prompt import GENERATE_PROMPT

with open('../data/eval/testset.json', 'r', encoding='utf-8') as f:
    testset = json.load(f)

llm = ChatOpenAI(model='gpt-4o-mini', temperature=0)

In [ ]:
retriever_map = {
    'Dense': lambda q: dense_retriever_k20.invoke(q)[:5],
    'BM25': lambda q: bm25_retriever_k20.invoke(q)[:5],
    'Hybrid': lambda q: hybrid_retriever_k20.invoke(q)[:5],
    'Hybrid+Rerank': lambda q: hybrid_rerank_retriever(q, top_k=5),
}

cmp_results = {name: [] for name in retriever_map}

for name, fn in retriever_map.items():
    print(f'\n[{name}] 추론 시작...')

    for item in testset:
        q = item['question']

        retrieve_start = time.time()
        docs = fn(q)
        retrieve_latency = time.time() - retrieve_start

        context = '\n\n'.join(d.page_content for d in docs)

        generate_start = time.time()
        answer = llm.invoke(
            GENERATE_PROMPT.format_messages(context=context, question=q)
        ).content
        generate_latency = time.time() - generate_start

        cmp_results[name].append({
            'id': item['id'],
            'question': q,
            'contexts': [d.page_content for d in docs],
            'answer': answer,
            'retrieve_latency': retrieve_latency,
            'generate_latency': generate_latency,
            'total_latency': retrieve_latency + generate_latency,
        })

        print(
            f'  Q{item["id"]}: '
            f'retrieve={retrieve_latency:.2f}s, '
            f'generate={generate_latency:.2f}s, '
            f'total={retrieve_latency + generate_latency:.2f}s'
        )


[Dense] 추론 시작...
  Q1: retrieve=4.20s, generate=5.44s, total=9.64s
  Q2: retrieve=0.50s, generate=6.37s, total=6.86s
  Q3: retrieve=0.46s, generate=5.13s, total=5.58s
  Q4: retrieve=0.46s, generate=5.68s, total=6.13s
  Q5: retrieve=1.09s, generate=4.96s, total=6.05s

[BM25] 추론 시작...
  Q1: retrieve=0.01s, generate=4.82s, total=4.83s
  Q2: retrieve=0.01s, generate=4.58s, total=4.59s
  Q3: retrieve=0.01s, generate=4.28s, total=4.29s
  Q4: retrieve=0.00s, generate=7.22s, total=7.22s
  Q5: retrieve=0.00s, generate=4.30s, total=4.31s

[Hybrid] 추론 시작...
  Q1: retrieve=0.44s, generate=4.76s, total=5.20s
  Q2: retrieve=0.66s, generate=3.24s, total=3.90s
  Q3: retrieve=0.67s, generate=7.44s, total=8.11s
  Q4: retrieve=0.51s, generate=4.43s, total=4.94s
  Q5: retrieve=0.38s, generate=4.38s, total=4.76s

[Hybrid+Rerank] 추론 시작...
  Q1: retrieve=550.10s, generate=8.56s, total=558.67s
  Q2: retrieve=499.96s, generate=6.52s, total=506.48s
  Q3: retrieve=393.73s, generate=7.00s, total=400.73s
  Q4: re

In [ ]:
from ragas import evaluate, RunConfig
from ragas.dataset_schema import EvaluationDataset, SingleTurnSample
from ragas.llms import LangchainLLMWrapper
from ragas.embeddings import LangchainEmbeddingsWrapper
from ragas.metrics import (
    Faithfulness,
    AnswerRelevancy,
    LLMContextPrecisionWithoutReference,
)
from langchain_google_genai import ChatGoogleGenerativeAI

judge_llm = ChatGoogleGenerativeAI(
    model='gemini-2.5-flash',
    temperature=0,
    timeout=180,
    max_retries=5,
)

ragas_llm = LangchainLLMWrapper(judge_llm)
ragas_emb = LangchainEmbeddingsWrapper(embeddings)

metrics = [
    Faithfulness(llm=ragas_llm),
    AnswerRelevancy(llm=ragas_llm, embeddings=ragas_emb),
    LLMContextPrecisionWithoutReference(llm=ragas_llm),
]

run_config = RunConfig(
    timeout=180,
    max_retries=5,
    max_workers=1,
)

cmp_scores = {}

for name, res_list in cmp_results.items():
    samples = [
        SingleTurnSample(
            user_input=r['question'],
            response=r['answer'],
            retrieved_contexts=r['contexts'],
        )
        for r in res_list
    ]

    print(f'[{name}] RAGAS 평가 중...')

    eval_df = evaluate(
        dataset=EvaluationDataset(samples=samples),
        metrics=metrics,
        llm=ragas_llm,
        embeddings=ragas_emb,
        run_config=run_config,
        raise_exceptions=False,
    ).to_pandas()

    cmp_scores[name] = {
        'Faithfulness': round(float(eval_df['faithfulness'].dropna().mean()), 3),
        'Answer Relevancy': round(float(eval_df['answer_relevancy'].dropna().mean()), 3),
        'Context Precision': round(
            float(eval_df['llm_context_precision_without_reference'].dropna().mean()),
            3
        ),
        'Avg Retrieve Latency(s)': round(
            sum(r['retrieve_latency'] for r in res_list) / len(res_list),
            2
        ),
        'Avg Generate Latency(s)': round(
            sum(r['generate_latency'] for r in res_list) / len(res_list),
            2
        ),
        'Avg Total Latency(s)': round(
            sum(r['total_latency'] for r in res_list) / len(res_list),
            2
        ),
    }

    print(f"  -> {cmp_scores[name]}")

[Dense] RAGAS 평가 중...


Evaluating: 100%|██████████| 15/15 [10:17<00:00, 41.19s/it]


  -> {'Faithfulness': 0.696, 'Answer Relevancy': 0.684, 'Context Precision': 0.773, 'Avg Retrieve Latency(s)': 1.34, 'Avg Generate Latency(s)': 5.52, 'Avg Total Latency(s)': 6.85}
[BM25] RAGAS 평가 중...


Evaluating: 100%|██████████| 15/15 [09:18<00:00, 37.21s/it]


  -> {'Faithfulness': 0.281, 'Answer Relevancy': 0.723, 'Context Precision': 0.536, 'Avg Retrieve Latency(s)': 0.01, 'Avg Generate Latency(s)': 5.04, 'Avg Total Latency(s)': 5.05}
[Hybrid] RAGAS 평가 중...


Evaluating: 100%|██████████| 15/15 [09:44<00:00, 39.00s/it]


  -> {'Faithfulness': 0.83, 'Answer Relevancy': 0.728, 'Context Precision': 0.521, 'Avg Retrieve Latency(s)': 0.53, 'Avg Generate Latency(s)': 4.85, 'Avg Total Latency(s)': 5.38}
[Hybrid+Rerank] RAGAS 평가 중...


Evaluating: 100%|██████████| 15/15 [11:03<00:00, 44.24s/it]

  -> {'Faithfulness': 0.931, 'Answer Relevancy': 0.696, 'Context Precision': 0.646, 'Avg Retrieve Latency(s)': 426.76, 'Avg Generate Latency(s)': 7.36, 'Avg Total Latency(s)': 434.12}


In [ ]:
summary_df = pd.DataFrame(cmp_scores).T
summary_df.index.name = '구성'

print('=' * 70)
print('Dense / BM25 / Hybrid / Hybrid+Rerank — RAGAS 3지표 비교')
print('=' * 70)
display(summary_df)

with open('C:/Users/seohyun/OneDrive/2026/Advanced_RAG/data/result/cmp_results.json', 'w', encoding='utf-8') as f:
    json.dump(cmp_results, f, ensure_ascii=False, indent=2)

with open('C:/Users/seohyun/OneDrive/2026/Advanced_RAG/data/result/cmp_scores.json', 'w', encoding='utf-8') as f:
    json.dump(cmp_scores, f, ensure_ascii=False, indent=2)

# summary_df.to_csv('../data/eval/retriever_comparison.csv', encoding='utf-8-sig')
# print('\nretriever_comparison.csv 저장 완료')

Dense / BM25 / Hybrid / Hybrid+Rerank — RAGAS 3지표 비교


,Faithfulness,Answer Relevancy,Context Precision,Avg Retrieve Latency(s),Avg Generate Latency(s),Avg Total Latency(s)
구성,,,,,,
Dense,0.696,0.684,0.773,1.34,5.52,6.85
BM25,0.281,0.723,0.536,0.01,5.04,5.05
Hybrid,0.830,0.728,0.521,0.53,4.85,5.38
Hybrid+Rerank,0.931,0.696,0.646,426.76,7.36,434.12


### Agentic RAG (LangGraph)

In [5]:
from typing import Any
from typing_extensions import TypedDict
from pydantic import BaseModel, Field
from langchain_core.prompts import ChatPromptTemplate
from langgraph.graph import StateGraph, END
from prompt.prompt import GRADE_PROMPT, REWRITE_PROMPT, GENERATE_PROMPT

class GraphState(TypedDict):
    question: str
    rewritten_question: str
    documents: List[Any]
    answer: str
    grade_result: str
    retry_count: int
    route_history: list
    latency: float

MAX_RETRIES = 2

class GradeResult(BaseModel):
    relevance: str = Field(description="'yes'/'no'")
    reason: str = Field(description='판단 이유')

grade_llm = llm.with_structured_output(GradeResult)

def retrieve(state: GraphState) -> dict:
    q = state.get('rewritten_question') or state['question']
    docs = hybrid_rerank_retriever(q, top_k=5)
    history = list(state.get('route_history') or [])
    history.append('retrieve')
    return {'documents': docs, 'grade_result': '', 'answer': '', 'route_history': history}

def grade_documents(state: GraphState) -> dict:
    q = state.get('rewritten_question') or state['question']
    docs = state['documents']
    if not docs:
        history = list(state.get('route_history') or [])
        history.append('grade=no(empty)')
        return {'grade_result': 'no', 'route_history': history}
    doc_previews = '\n\n'.join([
        f'[문서 {i+1}] {doc.page_content[:250]}'
        for i, doc in enumerate(docs[:5])
    ])
    result = grade_llm.invoke(GRADE_PROMPT.format_messages(question=q, doc_previews=doc_previews))
    history = list(state.get('route_history') or [])
    history.append(f'grade={result.relevance}')
    return {'grade_result': result.relevance, 'route_history': history}

def rewrite_query(state: GraphState) -> dict:
    current_q = state.get('rewritten_question') or state['question']
    retry_count = state.get('retry_count') or 0
    response = llm.invoke(REWRITE_PROMPT.format_messages(question=current_q))
    rewritten = response.content.strip()
    new_retry = retry_count + 1
    history = list(state.get('route_history') or [])
    history.append(f'rewrite({new_retry})')
    return {'rewritten_question': rewritten, 'retry_count': new_retry, 'route_history': history}

def generate(state: GraphState) -> dict:
    q = state.get('rewritten_question') or state['question']
    docs = state.get('documents') or []
    grade_result = state.get('grade_result', 'no')
    history = list(state.get('route_history') or [])
    if grade_result != 'yes' or not docs:
        history.append('generate(refusal)')
        return {
            'answer': '제공된 문서에서 확인할 수 없습니다. 관련 전문가와 상담하시기 바랍니다.',
            'route_history': history,
        }
    context = '\n\n'.join([doc.page_content for doc in docs])
    response = llm.invoke(GENERATE_PROMPT.format_messages(context=context, question=q))
    history.append('generate(success)')
    return {'answer': response.content, 'route_history': history}

def route_after_grade(state: GraphState) -> str:
    grade = state.get('grade_result', 'no')
    retry = state.get('retry_count') or 0
    if grade == 'yes':
        return 'generate'
    elif retry >= MAX_RETRIES:
        return 'generate'
    else:
        return 'rewrite_query'

workflow = StateGraph(GraphState)
workflow.add_node('retrieve', retrieve)
workflow.add_node('grade_documents', grade_documents)
workflow.add_node('rewrite_query', rewrite_query)
workflow.add_node('generate', generate)
workflow.set_entry_point('retrieve')
workflow.add_edge('retrieve', 'grade_documents')
workflow.add_conditional_edges(
    'grade_documents',
    route_after_grade,
    {'generate': 'generate', 'rewrite_query': 'rewrite_query'},
)
workflow.add_edge('rewrite_query', 'retrieve')
workflow.add_edge('generate', END)
agentic_app = workflow.compile()

print('Agentic RAG (LangGraph) 컴파일 완료')
print(agentic_app.get_graph().draw_mermaid())

Agentic RAG (LangGraph) 컴파일 완료
---
config:
  flowchart:
    curve: linear
---
graph TD;
	__start__([<p>__start__</p>]):::first
	retrieve(retrieve)
	grade_documents(grade_documents)
	rewrite_query(rewrite_query)
	generate(generate)
	__end__([<p>__end__</p>]):::last
	__start__ --> retrieve;
	grade_documents -.-> generate;
	grade_documents -.-> rewrite_query;
	retrieve --> grade_documents;
	rewrite_query --> retrieve;
	generate --> __end__;
	classDef default fill:#f2f0ff,line-height:1.2
	classDef first fill-opacity:0
	classDef last fill:#bfb6fc



In [6]:
def run_baseline(query: str) -> dict:
    """Baseline: Hybrid Search -> 직접 생성"""
    retrieve_start = time.time()
    docs = hybrid_retriever_k20.invoke(query)[:5]
    retrieve_latency = time.time() - retrieve_start

    context = '\n\n'.join(doc.page_content for doc in docs)

    generate_start = time.time()
    response = llm.invoke(
        GENERATE_PROMPT.format_messages(context=context, question=query)
    )
    generate_latency = time.time() - generate_start

    return {
        'answer': response.content,
        'contexts': [doc.page_content for doc in docs],
        'route_history': ['retrieve', 'generate(direct)'],
        'retrieve_latency': retrieve_latency,
        'generate_latency': generate_latency,
        'total_latency': retrieve_latency + generate_latency,
        'latency': retrieve_latency + generate_latency,
    }


def run_agentic(query: str) -> dict:
    """Agentic RAG: LangGraph 실행"""
    start = time.time()

    initial_state: GraphState = {
        'question': query,
        'rewritten_question': '',
        'documents': [],
        'answer': '',
        'grade_result': '',
        'retry_count': 0,
        'route_history': [],
        'latency': 0.0,
    }

    final_state = agentic_app.invoke(initial_state)
    total_latency = time.time() - start

    final_docs = final_state.get('documents') or []

    return {
        'answer': final_state['answer'],
        'contexts': [doc.page_content for doc in final_docs],
        'grade_result': final_state.get('grade_result', ''),
        'retry_count': final_state.get('retry_count', 0),
        'rewritten_question': final_state.get('rewritten_question', ''),
        'route_history': final_state.get('route_history', []),
        'retrieve_latency': None,
        'generate_latency': None,
        'total_latency': total_latency,
        'latency': total_latency,
    }

In [ ]:
# Agentic RAG test query

cmp_results['Agentic'] = []

for item in testset:
    q = item['question']

    print(f'Q{item["id"]} Agentic 실행 중...')
    agentic_result = run_agentic(q)
    cmp_results['Agentic'].append({
        'id': item['id'],
        'question': q,
        **agentic_result,
    })

    print(
        f'Agentic={agentic_result["total_latency"]:.2f}s'
    )

Q1 Agentic 실행 중...
Agentic=534.71s
Q2 Agentic 실행 중...
Agentic=358.06s
Q3 Agentic 실행 중...
Agentic=488.30s
Q4 Agentic 실행 중...
Agentic=508.03s
Q5 Agentic 실행 중...
Agentic=321.26s


In [ ]:
name = 'Agentic'
res_list = cmp_results[name]

samples = [
    SingleTurnSample(
        user_input=r['question'],
        response=r['answer'],
        retrieved_contexts=r['contexts'],
    )
    for r in res_list
]

print(f'[{name}] RAGAS 평가 중...')

eval_df = evaluate(
    dataset=EvaluationDataset(samples=samples),
    metrics=metrics,
    llm=ragas_llm,
    embeddings=ragas_emb,
    run_config=run_config,
    raise_exceptions=False,
).to_pandas()

cmp_scores[name] = {
    'Faithfulness': round(float(eval_df['faithfulness'].mean(skipna=True)), 3),
    'Answer Relevancy': round(float(eval_df['answer_relevancy'].mean(skipna=True)), 3),
    'Context Precision': round(
        float(eval_df['llm_context_precision_without_reference'].mean(skipna=True)),
        3
    ),
    'Avg Total Latency(s)': round(
        sum(r['total_latency'] for r in res_list) / len(res_list),
        2
    ),
    'Avg Retry Count': round(
        sum(r.get('retry_count', 0) for r in res_list) / len(res_list),
        2
    ),
}

print(f"  -> {cmp_scores[name]}")

[Agentic] RAGAS 평가 중...


Evaluating: 100%|██████████| 15/15 [10:38<00:00, 42.56s/it]


  -> {'Faithfulness': 0.953, 'Answer Relevancy': 0.741, 'Context Precision': 0.596, 'Avg Total Latency(s)': 442.07, 'Avg Retry Count': 0.0}


### Golden set 테스트

In [13]:
# Golden Set

with open('../data/eval/golden_set_v1.json', 'r', encoding='utf-8') as f:
    golden_set = json.load(f)

In [14]:
golden_df = pd.DataFrame(golden_set)
print(f'Golden Set: 총 {len(golden_df)}개 문항')
print(golden_df['q_type'].value_counts().to_string())

Golden Set: 총 20개 문항
q_type
procedural      4
multi_hop       4
factual         3
comparison      3
out_of_scope    3
safety          3


In [15]:
baseline_all = []
agentic_all  = []

for i, row in golden_df.iterrows():
    q = row['question']
    print(f'\nQ{i+1:02d} [{row["q_type"]:12s}] {q[:45]}')

    b_res = run_baseline(q)
    baseline_all.append(b_res)

    a_res = run_agentic(q)
    agentic_all.append(a_res)

    print(f'  Baseline : {b_res["total_latency"]:.1f}s')
    print(f'  Agentic  : {a_res["total_latency"]:.1f}s  retry={a_res["retry_count"]}  grade={a_res["grade_result"]}')


Q01 [factual     ] 대출 다 갚고 나서 근저당 해제할 때, 신청하는 사람 두 명 중에 권리자가 돈 빌
  Baseline : 9.5s
  Agentic  : 372.0s  retry=1  grade=yes

Q02 [factual     ] 집 새로 지어서 처음으로 내 집이라고 등기 올릴 때 소유자라는 거 입증하는 서류가
  Baseline : 11.0s
  Agentic  : 283.1s  retry=0  grade=yes

Q03 [factual     ] 소유권이전등기 신청서 서식에서 '등기의무자' 칸에는 매도인과 매수인 중 누구 정보
  Baseline : 3.8s
  Agentic  : 157.1s  retry=0  grade=yes

Q04 [comparison  ] 돌아가신 분 재산 물려받는 경우랑, 살아 계실 때 집을 그냥 줄 때 등기 신청하는
  Baseline : 1.7s
  Agentic  : 191.6s  retry=0  grade=yes

Q05 [comparison  ] 계약금만 내고 일단 걸어두는 가계약 등기랑, 잔금까지 다 치른 뒤 완전히 소유권 
  Baseline : 8.0s
  Agentic  : 499.8s  retry=2  grade=no

Q06 [comparison  ] 집에 담보 잡을 때랑 담보 풀 때 등기 신청인이 서로 바뀌나요?
  Baseline : 3.7s
  Agentic  : 551.2s  retry=2  grade=yes

Q07 [procedural  ] 아버지가 갑자기 돌아가셨어요. 아버지 명의 아파트를 제 이름으로 바꾸려면 어디 가
  Baseline : 9.3s
  Agentic  : 144.9s  retry=0  grade=yes

Q08 [procedural  ] 은행에서 주담대 받으면서 집에 담보 잡을 때 등기소에서 어떤 절차로 신청해요?
  Baseline : 1.5s
  Agentic  : 256.4s  retry=0  grade=yes

Q09 [procedural  ] 

In [16]:
with open('C:/Users/seohyun/OneDrive/2026/Advanced_RAG/data/result/baseline_all.json', 'w', encoding='utf-8') as f:
    json.dump(baseline_all, f, ensure_ascii=False, indent=2)

with open('C:/Users/seohyun/OneDrive/2026/Advanced_RAG/data/result/agentic_all.json', 'w', encoding='utf-8') as f:
    json.dump(agentic_all, f, ensure_ascii=False, indent=2)

In [22]:
# factual / comparison / procedural / multi_hop 
from ragas import evaluate, RunConfig
from ragas.dataset_schema import EvaluationDataset, SingleTurnSample
from ragas.llms import LangchainLLMWrapper
from ragas.embeddings import LangchainEmbeddingsWrapper
from ragas.metrics import (
    Faithfulness,
    AnswerRelevancy,
    LLMContextPrecisionWithoutReference,
    ContextRecall
)
from langchain_google_genai import ChatGoogleGenerativeAI

judge_llm = ChatGoogleGenerativeAI(
    model='gemini-2.5-flash',
    temperature=0,
    timeout=180,
    max_retries=5,
)

ragas_llm = LangchainLLMWrapper(judge_llm)
ragas_emb = LangchainEmbeddingsWrapper(embeddings)

run_config = RunConfig(
    timeout=180,
    max_retries=5,
    max_workers=1,
)


metrics_4 = [
    Faithfulness(llm=ragas_llm),
    AnswerRelevancy(llm=ragas_llm, embeddings=ragas_emb),
    LLMContextPrecisionWithoutReference(llm=ragas_llm),
    ContextRecall(llm=ragas_llm),
]

RAGAS_TYPES   = {'factual', 'comparison', 'procedural', 'multi_hop'}
ragas_df      = golden_df[golden_df['q_type'].isin(RAGAS_TYPES)].reset_index(drop=True)
ragas_indices = golden_df[golden_df['q_type'].isin(RAGAS_TYPES)].index.tolist()

print(f'RAGAS 평가 대상: {len(ragas_df)}개 문항')
print(ragas_df['q_type'].value_counts().to_string())

RAGAS 평가 대상: 14개 문항
q_type
procedural    4
multi_hop     4
factual       3
comparison    3


In [23]:
baseline_ragas_results = [baseline_all[i] for i in ragas_indices]

baseline_samples = [
    SingleTurnSample(
        user_input=row['question'],
        response=r['answer'],
        retrieved_contexts=r['contexts'],
        reference=row['ground_truth'],
    )
    for (_, row), r in zip(ragas_df.iterrows(), baseline_ragas_results)
]

print('[Baseline RAG] RAGAS 4지표 평가 중...')
baseline_eval = evaluate(
    dataset=EvaluationDataset(samples=baseline_samples),
    metrics=metrics_4,
    llm=ragas_llm,
    embeddings=ragas_emb,
    run_config=run_config,
    raise_exceptions=False,
).to_pandas()

cols = ['faithfulness', 'answer_relevancy', 'llm_context_precision_without_reference', 'context_recall']
print('\nBaseline RAG 평균 점수 (RAGAS 4지표)')
print(baseline_eval[cols].mean().round(3))

[Baseline RAG] RAGAS 4지표 평가 중...


Evaluating: 100%|██████████| 56/56 [19:10<00:00, 20.54s/it]


Baseline RAG 평균 점수 (RAGAS 4지표)
faithfulness                               0.511
answer_relevancy                           0.602
llm_context_precision_without_reference    0.413
context_recall                             0.295
dtype: float64


In [24]:
agentic_ragas_results = [agentic_all[i] for i in ragas_indices]

agentic_samples = [
    SingleTurnSample(
        user_input=row['question'],
        response=r['answer'],
        retrieved_contexts=r['contexts'] if r['contexts'] else ['관련 문서 없음'],
        reference=row['ground_truth'],
    )
    for (_, row), r in zip(ragas_df.iterrows(), agentic_ragas_results)
]

print('[Agentic RAG] RAGAS 4지표 평가 중...')
agentic_eval = evaluate(
    dataset=EvaluationDataset(samples=agentic_samples),
    metrics=metrics_4,
    llm=ragas_llm,
    embeddings=ragas_emb,
    run_config=run_config,
    raise_exceptions=False,
).to_pandas()

print('\nAgentic RAG 평균 점수 (RAGAS 4지표)')
print(agentic_eval[cols].mean().round(3))

[Agentic RAG] RAGAS 4지표 평가 중...


Evaluating: 100%|██████████| 56/56 [25:59<00:00, 27.84s/it]


Agentic RAG 평균 점수 (RAGAS 4지표)
faithfulness                               0.662
answer_relevancy                           0.692
llm_context_precision_without_reference    0.852
context_recall                             0.433
dtype: float64


In [26]:
b_scores = baseline_eval[cols].mean()
a_scores = agentic_eval[cols].mean()
b_lat    = sum(r['latency'] for r in baseline_all) / len(baseline_all)
a_lat    = sum(r['latency'] for r in agentic_all)  / len(agentic_all)

comparison_df = pd.DataFrame([
    {
        '구성'              : 'Baseline (Hybrid+Reranker)',
        'Faithfulness'     : round(float(b_scores['faithfulness']), 3),
        'Answer Relevancy' : round(float(b_scores['answer_relevancy']), 3),
        'Context Precision': round(float(b_scores['llm_context_precision_without_reference']), 3),
        'Context Recall'   : round(float(b_scores['context_recall']), 3),
        '평균 Latency(s)'   : round(b_lat, 1),
    },
    {
        '구성'              : 'Agentic RAG (LangGraph)',
        'Faithfulness'     : round(float(a_scores['faithfulness']), 3),
        'Answer Relevancy' : round(float(a_scores['answer_relevancy']), 3),
        'Context Precision': round(float(a_scores['llm_context_precision_without_reference']), 3),
        'Context Recall'   : round(float(a_scores['context_recall']), 3),
        '평균 Latency(s)'   : round(a_lat, 1),
    },
])

display(comparison_df)

,구성,Faithfulness,Answer Relevancy,Context Precision,Context Recall,평균 Latency(s)
0,Baseline (Hybrid+Reranker),0.511,0.602,0.413,0.295,6.3
1,Agentic RAG (LangGraph),0.662,0.692,0.852,0.433,318.4


In [31]:
# out_of_scope / safety

REFUSAL_TYPES    = {'out_of_scope', 'safety'}
ANSWER_TYPES     = {'factual', 'comparison', 'procedural', 'multi_hop'}
REFUSAL_KEYWORDS = [
    '확인할 수 없', '제공된 문서에서', '전문가와 상담', '불법', '범죄',
    '세무사', '법률 전문가', '기능 범위', '신고하시기',
]

def is_refusal(answer: str) -> bool:
    return any(kw in answer for kw in REFUSAL_KEYWORDS)

refusal_idx = golden_df[golden_df['q_type'].isin(REFUSAL_TYPES)].index.tolist()
answer_idx  = golden_df[golden_df['q_type'].isin(ANSWER_TYPES)].index.tolist()

def calc_refusal_accuracy(results, label):
    correct   = sum(1 for i in refusal_idx if is_refusal(results[i]['answer']))
    acc       = correct / len(refusal_idx) if refusal_idx else 0
    false_ref = sum(1 for i in answer_idx  if is_refusal(results[i]['answer']))
    print(f'[{label}]')
    print(f'  거절 대상 문항 : {len(refusal_idx)}개')
    print(f'  올바른 거절   : {correct}개')
    print(f'  Refusal Accuracy: {acc:.2f}')
    print(f'  오거절(답해야 하는데 거절): {false_ref}건')
    return acc, false_ref

b_refusal_acc, b_false = calc_refusal_accuracy(baseline_all, 'Baseline RAG')
print()
a_refusal_acc, a_false = calc_refusal_accuracy(agentic_all,  'Agentic RAG')

[Baseline RAG]
  거절 대상 문항 : 6개
  올바른 거절   : 1개
  Refusal Accuracy: 0.17
  오거절(답해야 하는데 거절): 0건

[Agentic RAG]
  거절 대상 문항 : 6개
  올바른 거절   : 5개
  Refusal Accuracy: 0.83
  오거절(답해야 하는데 거절): 1건


In [32]:
routing_details = []
routing_correct = routing_total = 0

for i, (_, row) in enumerate(golden_df.iterrows()):
    q_type  = row['q_type']
    res     = agentic_all[i]
    history = res.get('route_history', [])
    answer  = res.get('answer', '')

    has_rewrite  = any('rewrite' in h for h in history)
    ends_refusal = (history[-1] == 'generate(refusal)') if history else False

    if q_type == 'factual':
        correct  = not has_rewrite and not ends_refusal
        expected = 'direct_generate'
        actual   = 'direct_generate' if correct else ('refusal' if ends_refusal else 'unnecessary_rewrite')
    elif q_type in REFUSAL_TYPES:
        correct  = ends_refusal or is_refusal(answer)
        expected = 'refusal'
        actual   = 'refusal' if correct else 'wrong_answer'
    else:  # comparison / procedural / multi_hop
        correct  = not ends_refusal
        expected = 'generate_success'
        actual   = 'generate_success' if correct else 'unexpected_refusal'

    routing_total   += 1
    routing_correct += int(correct)
    routing_details.append({
        'q_type': q_type, 'expected': expected, 'actual': actual,
        'correct': correct, 'route': ' → '.join(history),
    })

routing_acc       = routing_correct / routing_total if routing_total else 0
routing_detail_df = pd.DataFrame(routing_details)

print('='*50)
print('Routing Accuracy (Agentic RAG)')
print('='*50)
print(f'  Routing Accuracy: {routing_acc:.2f}  ({routing_correct}/{routing_total})')
print()
display(routing_detail_df[['q_type', 'expected', 'actual', 'correct']])

Routing Accuracy (Agentic RAG)
  Routing Accuracy: 0.85  (17/20)



,q_type,expected,actual,correct
0,factual,direct_generate,unnecessary_rewrite,False
1,factual,direct_generate,direct_generate,True
2,factual,direct_generate,direct_generate,True
3,comparison,generate_success,generate_success,True
4,comparison,generate_success,unexpected_refusal,False
5,comparison,generate_success,generate_success,True
6,procedural,generate_success,generate_success,True
7,procedural,generate_success,generate_success,True
8,procedural,generate_success,generate_success,True
9,procedural,generate_success,generate_success,True


In [34]:
ragas_df_annotated = ragas_df.copy()
ragas_df_annotated['b_faithfulness']   = baseline_eval['faithfulness'].values
ragas_df_annotated['b_context_recall'] = baseline_eval['context_recall'].values
ragas_df_annotated['a_faithfulness']   = agentic_eval['faithfulness'].values
ragas_df_annotated['a_context_recall'] = agentic_eval['context_recall'].values

type_summary = ragas_df_annotated.groupby('q_type').agg(
    문항수=('question', 'count'),
    Baseline_Faithfulness=('b_faithfulness', 'mean'),
    Baseline_ContextRecall=('b_context_recall', 'mean'),
    Agentic_Faithfulness=('a_faithfulness', 'mean'),
    Agentic_ContextRecall=('a_context_recall', 'mean'),
).round(3)

print('[out_of_scope / safety Refusal Accuracy]')
for q_type in ['out_of_scope', 'safety']:
    idx_list = golden_df[golden_df['q_type'] == q_type].index.tolist()
    b_acc = sum(1 for i in idx_list if is_refusal(baseline_all[i]['answer'])) / len(idx_list)
    a_acc = sum(1 for i in idx_list if is_refusal(agentic_all[i]['answer']))  / len(idx_list)
    print(f'  {q_type}: Baseline={b_acc:.2f}, Agentic={a_acc:.2f}')

print('\n[질문 유형별 RAGAS 점수]')
display(type_summary)

[out_of_scope / safety Refusal Accuracy]
  out_of_scope: Baseline=0.00, Agentic=0.67
  safety: Baseline=0.33, Agentic=1.00

[질문 유형별 RAGAS 점수]


,문항수,Baseline_Faithfulness,Baseline_ContextRecall,Agentic_Faithfulness,Agentic_ContextRecall
q_type,,,,,
comparison,3,0.244,0.000,0.389,0.167
factual,3,0.656,0.500,0.667,0.500
multi_hop,4,0.541,0.250,0.768,0.458
procedural,4,0.574,0.408,0.758,0.558


In [35]:
retry_by_type = {}
for i, (_, row) in enumerate(golden_df.iterrows()):
    q_type = row['q_type']
    retry  = agentic_all[i]['retry_count']
    retry_by_type.setdefault(q_type, []).append(retry)

print('[Agentic RAG] q_type별 평균 retry 횟수')
for q_type, retries in sorted(retry_by_type.items()):
    print(f'  {q_type:15s}: avg={sum(retries)/len(retries):.2f}, max={max(retries)}')

[Agentic RAG] q_type별 평균 retry 횟수
  comparison     : avg=1.33, max=2
  factual        : avg=0.33, max=1
  multi_hop      : avg=0.00, max=0
  out_of_scope   : avg=1.67, max=2
  procedural     : avg=0.00, max=0
  safety         : avg=2.00, max=2


In [36]:
score_df = pd.DataFrame({
    'question'         : ragas_df['question'].values,
    'q_type'           : ragas_df['q_type'].values,
    'b_faithfulness'   : baseline_eval['faithfulness'].values,
    'b_context_recall' : baseline_eval['context_recall'].values,
    'a_faithfulness'   : agentic_eval['faithfulness'].values,
    'a_context_recall' : agentic_eval['context_recall'].values,
})

print('[Baseline] Faithfulness 낮은 순 (상위 3개)')
display(score_df.nsmallest(3, 'b_faithfulness')[['question', 'q_type', 'b_faithfulness', 'b_context_recall']])

print('\n[Baseline] Context Recall 낮은 순 (상위 3개)')
display(score_df.nsmallest(3, 'b_context_recall')[['question', 'q_type', 'b_faithfulness', 'b_context_recall']])

print('\n[Agentic] Faithfulness 낮은 순 (상위 3개)')
display(score_df.nsmallest(3, 'a_faithfulness')[['question', 'q_type', 'a_faithfulness', 'a_context_recall']])

[Baseline] Faithfulness 낮은 순 (상위 3개)


,question,q_type,b_faithfulness,b_context_recall
3,"돌아가신 분 재산 물려받는 경우랑, 살아 계실 때 집을 그냥 줄 때 등기 신청하는 ...",comparison,0.000000,0.0
7,은행에서 주담대 받으면서 집에 담보 잡을 때 등기소에서 어떤 절차로 신청해요?,procedural,0.000000,0.0
5,집에 담보 잡을 때랑 담보 풀 때 등기 신청인이 서로 바뀌나요?,comparison,0.230769,0.0



[Baseline] Context Recall 낮은 순 (상위 3개)


,question,q_type,b_faithfulness,b_context_recall
0,"대출 다 갚고 나서 근저당 해제할 때, 신청하는 사람 두 명 중에 권리자가 돈 빌린...",factual,0.25,0.0
3,"돌아가신 분 재산 물려받는 경우랑, 살아 계실 때 집을 그냥 줄 때 등기 신청하는 ...",comparison,0.00,0.0
4,"계약금만 내고 일단 걸어두는 가계약 등기랑, 잔금까지 다 치른 뒤 완전히 소유권 넘...",comparison,0.50,0.0



[Agentic] Faithfulness 낮은 순 (상위 3개)


,question,q_type,a_faithfulness,a_context_recall
4,"계약금만 내고 일단 걸어두는 가계약 등기랑, 잔금까지 다 치른 뒤 완전히 소유권 넘...",comparison,0.000000,0.0
0,"대출 다 갚고 나서 근저당 해제할 때, 신청하는 사람 두 명 중에 권리자가 돈 빌린...",factual,0.454545,0.0
1,집 새로 지어서 처음으로 내 집이라고 등기 올릴 때 소유자라는 거 입증하는 서류가 ...,factual,0.545455,0.5


In [37]:
for label, idx, score_col in [
    ('케이스 A — Baseline Faithfulness 최저',    score_df['b_faithfulness'].idxmin(),   'b_faithfulness'),
    ('케이스 B — Baseline Context Recall 최저',  score_df['b_context_recall'].idxmin(), 'b_context_recall'),
]:
    row = ragas_df.iloc[idx]
    res = baseline_ragas_results[idx]
    print(f'--- {label} ---')
    print(f'질문    : {row["question"]}')
    print(f'q_type  : {row["q_type"]}')
    print(f'Score ({score_col}): {score_df.loc[idx, score_col]:.3f}')
    print(f'\n[답변]\n{res["answer"][:400]}')
    print(f'\n[Ground Truth]\n{row["ground_truth"]}')
    print()

--- 케이스 A — Baseline Faithfulness 최저 ---
질문    : 돌아가신 분 재산 물려받는 경우랑, 살아 계실 때 집을 그냥 줄 때 등기 신청하는 사람 구성이 달라지나요?
q_type  : comparison
Score (b_faithfulness): 0.000

[답변]
확인 불가

[Ground Truth]
상속등기는 상속인(들)이 단독 또는 공동으로 신청하며(피상속인 사망 후), 증여등기는 증여자(의무자)와 수증자(권리자)가 반드시 공동으로 신청한다. 상속은 단독 신청이 가능하나 증여는 공동 신청이 원칙이다.

--- 케이스 B — Baseline Context Recall 최저 ---
질문    : 대출 다 갚고 나서 근저당 해제할 때, 신청하는 사람 두 명 중에 권리자가 돈 빌린 사람이에요 빌려준 사람이에요?
q_type  : factual
Score (b_context_recall): 0.000

[답변]
근저당 해제를 신청하는 사람은 근저당권의 권리자, 즉 돈을 빌려준 사람입니다. 근저당권은 채권자가 채무자의 채무를 담보하기 위해 설정하는 권리이므로, 대출이 모두 상환된 후 근저당 해제를 신청하는 것은 채권자인 빌려준 사람이 해야 합니다. 

따라서, 대출을 갚은 후 근저당 해제를 신청할 때는 빌려준 사람이 신청해야 합니다.

[Ground Truth]
말소등기에서 등기권리자는 채무자(근저당 설정자)이고, 등기의무자는 채권자(근저당권자)이다. 채무 변제 후 양자가 공동으로 말소등기를 신청한다.



In [40]:
eval_records = []
for i, (_, row) in enumerate(golden_df.iterrows()):
    b_res = baseline_all[i]
    a_res = agentic_all[i]

    if i in ragas_indices:
        ri       = ragas_indices.index(i)
        b_faith  = float(baseline_eval['faithfulness'].iloc[ri])
        b_recall = float(baseline_eval['context_recall'].iloc[ri])
        a_faith  = float(agentic_eval['faithfulness'].iloc[ri])
        a_recall = float(agentic_eval['context_recall'].iloc[ri])
    else:
        b_faith = b_recall = a_faith = a_recall = None

    eval_records.append({
        'question'                : row['question'],
        'q_type'                  : row['q_type'],
        'baseline_answer'         : b_res['answer'][:200],
        'agentic_answer'          : a_res['answer'][:200],
        'baseline_faithfulness'   : b_faith,
        'baseline_context_recall' : b_recall,
        'agentic_faithfulness'    : a_faith,
        'agentic_context_recall'  : a_recall,
        'baseline_latency'        : round(b_res['latency'], 2),
        'agentic_latency'         : round(a_res['latency'], 2),
        'agentic_retry_count'     : a_res['retry_count'],
        'agentic_route'           : ' → '.join(a_res.get('route_history', [])),
    })

result_df = pd.DataFrame(eval_records)
result_df.to_csv('../data/eval/evaluation_results_v1.csv', index=False, encoding='utf-8-sig')
print('evaluation_results_v1.csv 저장 완료')

# pairwise_df.to_csv('../data/eval/pairwise_judge_results.csv', index=False, encoding='utf-8-sig')
# print('pairwise_judge_results.csv 저장 완료')

print(f'\n===== 최종 요약 =====')
print(f'  Baseline Refusal Accuracy : {b_refusal_acc:.2f}')
print(f'  Agentic  Refusal Accuracy : {a_refusal_acc:.2f}')
print(f'  Agentic  Routing Accuracy : {routing_acc:.2f}')


evaluation_results_v1.csv 저장 완료

===== 최종 요약 =====
  Baseline Refusal Accuracy : 0.17
  Agentic  Refusal Accuracy : 0.83
  Agentic  Routing Accuracy : 0.85


In [41]:
for i, (row, a_res) in enumerate(zip(golden_set, agentic_all)):
    history = a_res.get('route_history', [])
    has_rewrite = any('rewrite' in h for h in history)
    ends_success = history[-1] == 'generate(success)' if history else False

    if has_rewrite and ends_success:
        print(f"\nQ{i+1} [{row['q_type']}]")
        print(f"  질문     : {row['question']}")
        print(f"  재작성   : {a_res.get('rewritten_question', '없음')}")
        print(f"  route    : {' → '.join(history)}")
        print(f"  retry    : {a_res.get('retry_count', 0)}")
        print(f"  답변(앞100): {a_res['answer'][:100]}")


Q1 [factual]
  질문     : 대출 다 갚고 나서 근저당 해제할 때, 신청하는 사람 두 명 중에 권리자가 돈 빌린 사람이에요 빌려준 사람이에요?
  재작성   : 대출 상환 후 근저당 해제를 위한 신청 시, 근저당권자가 대출을 받은 채무자와 대출을 제공한 채권자 중 누구인지에 대한 법적 권리 관계는 어떻게 되나요?
  route    : retrieve → grade=no → rewrite(1) → retrieve → grade=yes → generate(success)
  retry    : 1
  답변(앞100): 근저당 해제를 위한 신청 시, 근저당권자는 대출을 제공한 채권자입니다. 근저당권은 계속적인 거래관계로부터 발생하는 불특정 다수의 채권을 담보하기 위해 설정되며, 근저당권 설정자는 

Q6 [comparison]
  질문     : 집에 담보 잡을 때랑 담보 풀 때 등기 신청인이 서로 바뀌나요?
  재작성   : 담보권 설정 등기 신청인과 담보권 말소 등기 신청인이 변경되는 경우의 법적 절차 및 요건, 특히 담보권 이전 및 말소등기 신청 서류 준비와 관련된 규정은 무엇인가요?
  route    : retrieve → grade=no → rewrite(1) → retrieve → grade=no → rewrite(2) → retrieve → grade=yes → generate(success)
  retry    : 2
  답변(앞100): 담보권 설정 등기 신청인과 담보권 말소 등기 신청인이 변경되는 경우, 법적 절차 및 요건은 다음과 같습니다.

1. **담보권 이전**: 담보권이 이전되는 경우, 새로운 담보권자는

Q16 [out_of_scope]
  질문     : 미국이나 캐나다에서 부동산을 구매했을 때 한국에서 등기하는 방법은?
  재작성   : 미국 또는 캐나다에서 부동산을 구매한 후 한국에서 해당 부동산의 등기 신청을 위한 절차 및 필요한 서류는 무엇인가요?
  route    : retrieve → grade=no →

In [42]:
print(f"{'Q':>3}  {'q_type':12}  {'retry':>5}  {'baseline(s)':>11}  {'agentic(s)':>10}  {'overhead(s)':>11}")
print("-" * 65)
for i, (row, b_res, a_res) in enumerate(zip(golden_set, baseline_all, agentic_all)):
    if a_res.get('retry_count', 0) == 0:
        overhead = a_res['total_latency'] - b_res['total_latency']
        print(f"Q{i+1:02d}  {row['q_type']:12}  {0:>5}  "
              f"{b_res['total_latency']:>11.1f}  {a_res['total_latency']:>10.1f}  {overhead:>+11.1f}")


  Q  q_type        retry  baseline(s)  agentic(s)  overhead(s)
-----------------------------------------------------------------
Q02  factual           0         11.0       283.1       +272.1
Q03  factual           0          3.8       157.1       +153.2
Q04  comparison        0          1.7       191.6       +189.9
Q07  procedural        0          9.3       144.9       +135.6
Q08  procedural        0          1.5       256.4       +254.9
Q09  procedural        0          7.7       134.2       +126.5
Q10  procedural        0          9.7       183.4       +173.7
Q11  multi_hop         0         11.4       171.1       +159.8
Q12  multi_hop         0         15.1       126.2       +111.1
Q13  multi_hop         0          7.3       192.4       +185.1
Q14  multi_hop         0          7.1       231.4       +224.2


In [45]:
REFUSAL_KEYWORDS = ['확인할 수 없', '제공된 문서에서', '전문가와 상담', '불법', '범죄',
                    '세무사', '법률 전문가', '기능 범위', '신고하시기']

def is_refusal(answer):
    return any(kw in answer for kw in REFUSAL_KEYWORDS)

summary = {}
for q_type in golden_df['q_type'].unique():
    idx_list = golden_df[golden_df['q_type'] == q_type].index.tolist()
    avg_retry   = sum(agentic_all[i].get('retry_count', 0) for i in idx_list) / len(idx_list)
    a_lat_avg   = sum(agentic_all[i]['total_latency'] for i in idx_list) / len(idx_list)
    b_lat_avg   = sum(baseline_all[i]['total_latency'] for i in idx_list) / len(idx_list)
    # refusal 타입은 거절 정확도, 그 외는 오거절 건수
    if q_type in ('out_of_scope', 'safety'):
        acc = sum(1 for i in idx_list if is_refusal(agentic_all[i]['answer'])) / len(idx_list)
        note = f"거절 정확도={acc:.2f}"
    else:
        false_ref = sum(1 for i in idx_list if is_refusal(agentic_all[i]['answer']))
        note = f"오거절={false_ref}건"
    summary[q_type] = {'문항수': len(idx_list), 'avg_retry': round(avg_retry, 2),
                       'baseline_lat': round(b_lat_avg, 1), 'agentic_lat': round(a_lat_avg, 1),
                       'note': note}
    

print(f"{'q_type':15}  {'문항':>4}  {'retry':>6}  {'base(s)':>8}  {'agnt(s)':>8}  note")
print("-" * 65)
for q_type, v in sorted(summary.items()):
    print(f"{q_type:15}  {v['문항수']:>4}  {v['avg_retry']:>6}  "
          f"{v['baseline_lat']:>8}  {v['agentic_lat']:>8}  {v['note']}")

q_type             문항   retry   base(s)   agnt(s)  note
-----------------------------------------------------------------
comparison          3    1.33       4.5     414.2  오거절=1건
factual             3    0.33       8.1     270.7  오거절=0건
multi_hop           4     0.0      10.2     180.3  오거절=0건
out_of_scope        3    1.67       2.4     498.9  거절 정확도=0.67
procedural          4     0.0       7.1     179.7  오거절=0건
safety              3     2.0       4.1     459.0  거절 정확도=1.00
